# **RAG Architecture**

## **Loading document**

We are loading pdf using PyMuPDFLoader. It extracts tables as well as images from pdf. We are extracting tables in markdown format. We are loading whole pdf directly into memory.

In [1]:
# importing all libraries

from langchain_community.document_loaders import PyMuPDFLoader
from pprint import pprint as pp
from dotenv import load_dotenv
from uuid import uuid4

load_dotenv()

C:\Users\masan\AppData\Local\Temp\ipykernel_3172\1882016703.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


True

In [2]:
# document path
DOCUMENT_PATH = "../data/nexaai-company-policy.pdf"

# defining document loader
loader = PyMuPDFLoader(
    file_path = DOCUMENT_PATH,
    mode = "page",
    pages_delimiter = "", 
    extract_tables = "markdown"
)

In [3]:
docs = loader.load()

In [4]:
docs

[Document(metadata={'producer': 'Skia/PDF m127', 'creator': 'Chromium', 'creationdate': '2026-06-22T11:32:22+00:00', 'source': '../data/nexaai-company-policy.pdf', 'file_path': '../data/nexaai-company-policy.pdf', 'total_pages': 17, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-06-22T11:32:22+00:00', 'trapped': '', 'modDate': "D:20260622113222+00'00'", 'creationDate': "D:20260622113222+00'00'", 'page': 0}, page_content='Version: 2.0\nEffective Date: January 1, 2026\nDocument Owner: Chief Executive Officer\nLast Reviewed: June 1, 2026\nClassification: Internal — All Employees\nNexaAI Solutions is an AI-services company specializing in building custom artificial\nintelligence and machine learning systems for enterprise clients across banking, healthcare,\nretail, and logistics sectors. The company was founded in 2021 and is headquartered in Pune,\nIndia, with remote employees across the country. NexaAI delivers end-to-end AI solutions\nin

In [7]:
len(docs)

17

In [5]:
docs[0].metadata

{'producer': 'Skia/PDF m127',
 'creator': 'Chromium',
 'creationdate': '2026-06-22T11:32:22+00:00',
 'source': '../data/nexaai-company-policy.pdf',
 'file_path': '../data/nexaai-company-policy.pdf',
 'total_pages': 17,
 'format': 'PDF 1.4',
 'title': '',
 'author': '',
 'subject': '',
 'keywords': '',
 'moddate': '2026-06-22T11:32:22+00:00',
 'trapped': '',
 'modDate': "D:20260622113222+00'00'",
 'creationDate': "D:20260622113222+00'00'",
 'page': 0}

In [6]:
# checking table extraction
pp(docs[14].page_content)

('Level\n'
 'Notice Period\n'
 'Junior to Mid-level (< 3 years at NexaAI)\n'
 '30 days\n'
 'Senior / Lead / Manager (3+ years at NexaAI)\n'
 '60 days\n'
 'Principal / Architect / Director and above\n'
 '90 days\n'
 'NexaAI may waive the notice period at its discretion. Employees must not '
 'abruptly stop work\n'
 "without notice; doing so forfeits the final month's salary and may result in "
 'a negative reference.\n'
 'Within 45 days of the last working day, NexaAI will:\n'
 'Settle all outstanding salary and reimbursements.\n'
 'Process leave encashment of eligible Earned Leave balance.\n'
 'Issue Form 16 for the applicable tax year within the statutory deadline.\n'
 'Issue a relieving letter and experience certificate within 10 working days '
 'of the last day,\n'
 'provided no disciplinary action is pending.\n'
 'On or before the last working day, employees must return:\n'
 'Company laptop, charger, and accessories in working condition.\n'
 'Access badges and keys.\n'
 'Any physic

## **Docuemnt Chunking**

Using RecursiveCharacterTextSplitter for splitting documents.

In [8]:
# importing required libraries
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [9]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=250,
    separators=["\n\n", "\n", " ", ""]
)

chunks = splitter.split_documents(docs)

In [10]:
len(chunks)

41

In [11]:
chunks[10].metadata

{'producer': 'Skia/PDF m127',
 'creator': 'Chromium',
 'creationdate': '2026-06-22T11:32:22+00:00',
 'source': '../data/nexaai-company-policy.pdf',
 'file_path': '../data/nexaai-company-policy.pdf',
 'total_pages': 17,
 'format': 'PDF 1.4',
 'title': '',
 'author': '',
 'subject': '',
 'keywords': '',
 'moddate': '2026-06-22T11:32:22+00:00',
 'trapped': '',
 'modDate': "D:20260622113222+00'00'",
 'creationDate': "D:20260622113222+00'00'",
 'page': 4}

In [14]:
pp(chunks[11].page_content)

('The NexaAI leave year runs from January 1 to December 31. Leave balances '
 'reset on January 1\n'
 'each year. Employees who join mid-year will receive pro-rated leave balances '
 'based on the\n'
 'number of months remaining in the leave year.\n'
 'Entitlement: 18 days per calendar year, accrued at 1.5 days per month.\n'
 'Eligibility: Available after completing 3 months of employment.\n'
 'Application: Must be applied at least 3 working days in advance on the HR '
 'portal, subject to\n'
 'manager approval.\n'
 '3.4 Required Training During Onboarding\n'
 '3.5 Laptop and Equipment Policy\n'
 '4. Leave Policy\n'
 '4.1 Leave Year\n'
 '4.2 Types of Leave\n'
 '4.2.1 Earned Leave (EL)')


## **Document Embeddings and Vector Store**

We are using OpenAIEmbeddings and text-embedding-3-small model for embedding our chunks and queries. We are using chromadb as vector store

In [12]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

In [13]:
# initing embedding model
embeddings_function = OpenAIEmbeddings(
    model="text-embedding-3-small",
    dimensions=1024
    )


vector_store = Chroma(
    embedding_function=embeddings_function,
    collection_name="nexaai_collection",
)

In [14]:
uuids = [str(uuid4()) for _ in range(len(chunks))]

# adding documents in vector store
vector_store.add_documents(
    documents=chunks,
    ids=uuids,
    metadata={"hnsw:space": "cosine"}
)

['7e7a4880-cd7c-43f6-97db-1b3c940f6119',
 '4405548e-0450-4c91-8b5b-5fc08dac4d65',
 'bbbfe8dc-69a5-4793-ae31-f2e46323a416',
 'c7b075a3-3958-423b-91fe-e06342f21335',
 '4cb2e99b-3c1d-4e6c-bfcd-63573506b298',
 '0c7e501d-570b-4119-8fed-55489fb9f30c',
 'dbba15c4-4cce-4f7c-beed-5d15e560ddad',
 '4e61c857-2918-4dee-bbcb-a98a094e9285',
 'a6261f79-9652-418e-9ec1-67332c5090bd',
 'a7bafadb-cb39-48e5-8304-67f9534c3f73',
 '3224e98b-56ad-4715-bb81-9b0306d411bb',
 '8380c710-506c-4f0e-801f-8cbba211e0e4',
 'fe223c7b-23cf-4c88-a702-0d226744917a',
 'b66af5d8-3c88-485b-a7da-137839022a13',
 'a7e27b96-b10e-45e9-b2a9-e1739c2381ed',
 '8d59c8c9-9ba9-430d-8344-836c9dfbbbd5',
 '978a6203-e0b2-44d4-a131-8c04606d22d5',
 'db0e60e6-d512-4a47-a382-a0d7a3ffe658',
 '8db62905-16d0-493a-a159-200ca3dc5d71',
 'b8d30457-cc6d-42ce-9bd3-c4e504b6f7e6',
 '8b71917c-2bbe-418a-b6a1-bcf0a5052429',
 '24395cb3-5693-4720-ab5a-a6618d40dac4',
 'f598bea1-cb13-45cc-baa9-0e25d5ebf1c9',
 '3688837d-c617-4efa-8062-13b560b028ce',
 'cbdeaa9f-e856-

In [ ]:
# count of vectors present in vector store
vector_store._collection.count()

41

In [15]:
# searching with similarity score
vector_store.similarity_search_with_score("What is name of company", k=3)

[(Document(id='7e7a4880-cd7c-43f6-97db-1b3c940f6119', metadata={'keywords': '', 'moddate': '2026-06-22T11:32:22+00:00', 'file_path': '../data/nexaai-company-policy.pdf', 'total_pages': 17, 'subject': '', 'creationdate': '2026-06-22T11:32:22+00:00', 'creator': 'Chromium', 'trapped': '', 'modDate': "D:20260622113222+00'00'", 'source': '../data/nexaai-company-policy.pdf', 'title': '', 'page': 0, 'author': '', 'format': 'PDF 1.4', 'producer': 'Skia/PDF m127', 'creationDate': "D:20260622113222+00'00'"}, page_content='Version: 2.0\nEffective Date: January 1, 2026\nDocument Owner: Chief Executive Officer\nLast Reviewed: June 1, 2026\nClassification: Internal — All Employees\nNexaAI Solutions is an AI-services company specializing in building custom artificial\nintelligence and machine learning systems for enterprise clients across banking, healthcare,\nretail, and logistics sectors. The company was founded in 2021 and is headquartered in Pune,\nIndia, with remote employees across the country.

In [21]:
# creating retriever 
retriever = vector_store.as_retriever(
    search_type="mmr", search_kwargs={"k": 6, "lambda_mult": 0.25}
)

## **Augmentation and Generation**

Here we will define llm model and prompt template and final answer generation node. We will create sequansial chain using langchian's pipe operator

In [26]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

llm = ChatOpenAI(model="gpt-4o")

output_parser = StrOutputParser()

template = """You are a HR who has access of company policies, answers user's query regarding company policies from provided context only. Do not answer any other question which seems very outside your scope. Answer politely.
policy context:{context}

user's question: {question}"""

prompt = ChatPromptTemplate.from_template(template)

def join_retrieved_docs(docs):
    return "/n".join([doc.page_content for doc in docs])

# RAG Chain
rag_chain = {"context": retriever | join_retrieved_docs, "question":RunnablePassthrough()}| prompt | llm | output_parser


In [28]:
# invoking RAG chain
rag_chain.invoke("Hi my name is Yash. How can you help me?")

"Hello Yash! I'm here to assist you with any questions you may have about our company policies, especially regarding employee onboarding, health insurance, provident fund, and other HR-related policies from the context provided. If there's anything specific you'd like to know, feel free to ask!"